In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from tqdm import tqdm

In [ ]:
# Hyperparameters and Global Configurations
RANDOM_SEED = 42
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
NUM_WORKERS = 0
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_BACKBONE = 1e-4
WEIGHT_DECAY = 1e-4
HEAD_EPOCHS = 4
FINETUNE_EPOCHS = 3
MODEL_SAVE_PATH = "best_alzheimer_model.pth"
DATA_DIR = os.path.join("Alzeimer", "combined_images")

In [ ]:
print("Current working directory:", os.getcwd())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

In [ ]:
CLASS_NAMES = ["MildDemented", "ModerateDemented", "NonDemented", "VeryMildDemented"]
NUM_CLASSES = len(CLASS_NAMES)
class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASS_NAMES)}

records = []
SAMPLES_PER_CLASS = 1000

for cls_name in CLASS_NAMES:
    cls_folder = os.path.join(DATA_DIR, cls_name)
    if not os.path.exists(cls_folder):
        cls_folder = os.path.join("Alzeimer-prediction", DATA_DIR, cls_name)
    all_files = glob.glob(os.path.join(cls_folder, "*.jpg")) + glob.glob(os.path.join(cls_folder, "*.png"))
    all_files.sort()
    
    random.seed(RANDOM_SEED)
    selected_files = random.sample(all_files, min(SAMPLES_PER_CLASS, len(all_files)))
    for p in selected_files:
        records.append({
            "image_path": p,
            "class_name": cls_name,
            "diagnosis": class_to_idx[cls_name]
        })

df = pd.DataFrame(records)
print(f"Total dataset records: {len(df)}")
df.head()

In [ ]:
class_counts = df["diagnosis"].value_counts().sort_index()
print("Class Distribution:")
for i, count in enumerate(class_counts):
    print(f"{i} - {CLASS_NAMES[i]:18} : {count}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=[CLASS_NAMES[i] for i in class_counts.index], y=class_counts.values, palette="Blues_r")
plt.title("Alzheimer's Dataset Class Distribution", fontsize=14)
plt.xlabel("Cognitive Impairment Stage", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
class AlzheimerDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["diagnosis"])
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["diagnosis"],
    random_state=RANDOM_SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["diagnosis"],
    random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Split Summary:")
print("Training   :", len(train_df))
print("Validation :", len(val_df))
print("Testing    :", len(test_df))
print("Total      :", len(df))

In [ ]:
train_dataset = AlzheimerDataset(train_df, transform=train_transform)
val_dataset = AlzheimerDataset(val_df, transform=val_transform)
test_dataset = AlzheimerDataset(test_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("DataLoaders initialized successfully.")

In [ ]:
print("Loading pretrained EfficientNet-B0 backbone...")
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, NUM_CLASSES)
model = model.to(DEVICE)
print(model.classifier)

In [ ]:
for parameter in model.features.parameters():
    parameter.requires_grad = False

print("Backbone frozen. Trainable parameters in classifier head:")
for name, parameter in model.classifier.named_parameters():
    print(f"  {name}: {parameter.shape}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE_HEAD, weight_decay=WEIGHT_DECAY)
print("Loss function and Optimizer created.")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return running_loss / len(loader.dataset), acc, f1, all_labels, all_preds

In [ ]:
best_val_f1 = 0.0
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

print("--- Phase 1: Classifier Head Training ---")
for epoch in range(HEAD_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    print(f"Epoch {epoch+1}/{HEAD_EPOCHS+FINETUNE_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  --> Saved new best model ({MODEL_SAVE_PATH})")

In [ ]:
print("--- Phase 2: Fine-Tuning Top Feature Layers ---")
for param in model.features[-2:].parameters():
    param.requires_grad = True

ft_optimizer = torch.optim.AdamW([
    {"params": model.features[-2:].parameters(), "lr": LEARNING_RATE_BACKBONE},
    {"params": model.classifier.parameters(), "lr": 5e-4}
], weight_decay=WEIGHT_DECAY)

for epoch in range(FINETUNE_EPOCHS):
    curr_epoch = HEAD_EPOCHS + epoch + 1
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, ft_optimizer)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    print(f"Epoch {curr_epoch}/{HEAD_EPOCHS+FINETUNE_EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  --> Saved new best model ({MODEL_SAVE_PATH})")

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, len(val_losses)+1), val_losses, label="Val Loss", marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(train_accuracies)+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, len(val_accuracies)+1), val_accuracies, label="Val Accuracy", marker='o')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
print("Loading best saved model checkpoint...")
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))

test_loss, test_accuracy, test_f1, y_true, y_pred = evaluate(model, test_loader, criterion)
print("=" * 60)
print("TEST EVALUATION RESULTS")
print("=" * 60)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

In [ ]:
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Alzheimer's Disease Confusion Matrix", fontsize=14, pad=12)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

In [ ]:
qwk = cohen_kappa_score(y_true, y_pred)
print(f"Quadratic Weighted Kappa : {qwk:.4f}")

In [ ]:
def predict_image(image_path):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    tensor = val_transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        outputs = model(tensor)
        probabilities = torch.softmax(outputs, dim=1).squeeze().cpu().numpy()
        pred_idx = probabilities.argmax()
    print(f"Predicted Class: {CLASS_NAMES[pred_idx]} ({probabilities[pred_idx]*100:.2f}%)")
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {name:18}: {probabilities[i]*100:.2f}%")

In [ ]:
sample_test_img = test_df.iloc[0]["image_path"]
print(f"Testing prediction on: {sample_test_img}")
print(f"Actual Class: {test_df.iloc[0]['class_name']}")
predict_image(sample_test_img)